In [16]:
import os
import requests
from smolagents import Tool, CodeAgent, HfApiModel
from smolagents.default_tools import VisitWebpageTool

from duckduckgo_search import DDGS

class DuckDuckGoImageFetcher:
    name="get_ddg_image"
    description = "Get an image from DuckDuckGo"
    inputs = {}
    output_type = "image"

    def __init__(self, save_directory="images"):
        """
        Initialize the class with a directory to save images.
        :param save_directory: Directory where images will be saved.
        """
        self.save_directory = save_directory
        os.makedirs(self.save_directory, exist_ok=True)

    def search_image(self, query, max_results=1):
        """
        Search for images on DuckDuckGo.
        :param query: Search query (e.g., person's name).
        :param num_results: Number of image URLs to return.
        :return: List of image URLs.
        """
        results = DDGS().images(query, max_results=max_results)
        print(results)
        return [result["image"] for result in results] if results else []

    def download_image(self, url, filename):
        """
        Download an image from a URL and save it locally.
        :param url: URL of the image.
        :param filename: Name of the file to save.
        :return: Path to the saved image.
        """
        print ("Downloading image from ", url)
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            file_path = os.path.join(self.save_directory, filename)
            print(f"Saving image to: {file_path}")
            
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            print("Image downloaded successfully.")
            return file_path
        else:
            raise Exception(f"Failed to download image. Status code: {response.status_code}")

    def fetch_person_image(self, name):
        """
        Fetch the first image of a person by name.
        :param name: Name of the person to search for.
        :return: Path to the downloaded image.
        """
        print(f"Searching for images of {name}...")
        image_urls = self.search_image(name)
        if image_urls:
            print(f"Found image: {image_urls[0]}")
            filename = f"{name.replace(' ', '_')}.jpg"
            file_path = self.download_image(image_urls[0], filename)
            print(f"Image saved to: {file_path}")
            return file_path
        else:
            raise Exception(f"No images found for {name}.")

image_fetcher = DuckDuckGoImageFetcher()

einstein = image_fetcher.fetch_person_image("Albert Einstein")

agent = CodeAgent(
    tools = [image_fetcher, VisitWebpageTool()],
    model=HfApiModel(),
    additional_authorized_imports=["Pillow", "requests", "markdownify"], # "duckduckgo-search", 
    use_e2b_executor=True
)

agent.run(
    "Return me an image of a Albert Einstein. Directly use the image provided in your state.", additional_args={"get_ddg_image": image_fetcher()}
) # Asking to directly return the image from state tests that additional_args are properly sent to server.

agent

Searching for images of Albert Einstein...
[{'title': 'File:Albert Einstein Head.jpg', 'image': 'http://upload.wikimedia.org/wikipedia/commons/d/d3/Albert_Einstein_Head.jpg', 'thumbnail': 'https://tse1.mm.bing.net/th?id=OIP.l0lX0gL2ceT6ZwDATmjergHaJ3&pid=Api', 'url': 'http://commons.wikimedia.org/wiki/File:Albert_Einstein_Head.jpg', 'height': 4333, 'width': 3250, 'source': 'Bing'}]
Found image: http://upload.wikimedia.org/wikipedia/commons/d/d3/Albert_Einstein_Head.jpg
Saving image to: images\Albert_Einstein.jpg
Image downloaded successfully.
Image saved to: images\Albert_Einstein.jpg


AuthenticationException: API key is required, please visit the Team tab at https://e2b.dev/dashboard to get your API key. You can either set the environment variable `E2B_API_KEY` or you can pass it directly to the sandbox like Sandbox(api_key="e2b_...")